In [13]:
!nvidia-smi
import sys, torch
print("Python:", sys.version.split()[0], "| Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

Thu Sep 17 10:56:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             42W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [14]:
!pip install -q onnx onnxruntime timm jiwer unidecode omegaconf praat-textgrids \
    librosa speechbrain torchinfo torchprofile h5py transformers flashlight-text huggingface_hub pyyaml
from torchaudio.models.decoder import ctc_decoder
print("torchaudio ctc_decoder import OK")

torchaudio ctc_decoder import OK


In [15]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys, shutil
# ===================== EDIT THESE =====================
DRIVE_PROJECT = "/content/drive/MyDrive/silent_speech"   # your project folder on Drive
KENLM_SRC     = f"{DRIVE_PROJECT}/KenLM"                  # folder holding lm.bin + gaddy_lexicon.txt
# =====================================================
REPO_DIR = "/content/silent_speech"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/MatteoFasulo/silent_speech.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
!sed -i '/norm_layer=norm_layer,/d' {REPO_DIR}/architecture.py   # harmless if already fixed
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR); print("cwd:", os.getcwd())
LM_DIR = os.path.join(REPO_DIR, "KenLM")
if os.path.exists(KENLM_SRC) and not os.path.exists(LM_DIR):
    shutil.copytree(KENLM_SRC, LM_DIR)
print("KenLM ok:", os.path.exists(os.path.join(LM_DIR,"lm.bin")),
      os.path.exists(os.path.join(LM_DIR,"gaddy_lexicon.txt")))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
cwd: /content/silent_speech
KenLM ok: False False


In [16]:
from huggingface_hub import hf_hub_download
# If the repo is gated: from huggingface_hub import login; login("hf_xxx")  # then re-run
src = hf_hub_download(repo_id="PulpBio/TinyMyo",
                      filename="Silent_Speech/emg-to-text/tinymyo_ft_emg2text_epoch_157.pt")
CKPT = os.path.join(REPO_DIR, "tinymyo_ft_emg2text_epoch_157.pt")
shutil.copy(src, CKPT)
print("checkpoint:", CKPT, f"{os.path.getsize(CKPT)/1e6:.1f} MB")

checkpoint: /content/silent_speech/tinymyo_ft_emg2text_epoch_157.pt 18.3 MB


In [17]:
import glob, yaml
H5_REAL = f"{DRIVE_PROJECT}/emg_dataset.h5"
if not os.path.exists(H5_REAL):
    c = sorted(glob.glob("/content/drive/MyDrive/**/emg_dataset.h5", recursive=True))
    assert c, "emg_dataset.h5 not found on Drive"; H5_REAL = c[0]
os.environ["CKPT_DIR"]  = "/content/ckpts"; os.makedirs("/content/ckpts", exist_ok=True)
os.environ["DATA_PATH"] = DRIVE_PROJECT
patched = []
for cfg in glob.glob(os.path.join(REPO_DIR, "config", "*.y*ml")):
    with open(cfg) as f: d = yaml.safe_load(f) or {}
    if isinstance(d, dict) and "h5_path" in d:
        d["h5_path"] = H5_REAL
        with open(cfg, "w") as f: yaml.safe_dump(d, f, sort_keys=False)
        patched.append(os.path.basename(cfg))
print("h5:", H5_REAL, "| patched:", patched)   # expect ['data.yaml']

h5: /content/drive/MyDrive/silent_speech/emg_dataset.h5 | patched: ['data.yaml']


In [22]:
import os, glob, shutil
os.makedirs(LM_DIR, exist_ok=True)
lm = glob.glob("/content/drive/MyDrive/**/lm.bin", recursive=True) \
   + glob.glob("/content/silent_speech/**/lm.bin", recursive=True)
assert lm, "lm.bin not found"
dst = os.path.join(LM_DIR, "lm.bin")
if os.path.abspath(lm[0]) != os.path.abspath(dst):
    shutil.copy(lm[0], dst)
print("KenLM dir:", os.listdir(LM_DIR))   # expect BOTH: lm.bin and gaddy_derived_lexicon.txt

KenLM dir: ['gaddy_derived_lexicon.txt', 'lm.bin']


In [23]:
import os, glob, shutil
os.makedirs(LM_DIR, exist_ok=True)

def find(name):
    hits = glob.glob(f"/content/silent_speech/**/{name}", recursive=True) \
         + glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    return sorted(set(hits))

lm  = find("lm.bin")
der = find("gaddy_derived_lexicon.txt")
old = find("gaddy_lexicon.txt")
print("lm.bin           :", lm)
print("derived lexicon  :", der)
print("old lexicon      :", old)

assert lm, "lm.bin not found on repo or Drive — tell me where your KenLM lm.bin is."
shutil.copy(lm[0], os.path.join(LM_DIR, "lm.bin"))

if der:                                   # use the real derived lexicon if it exists
    shutil.copy(der[0], os.path.join(LM_DIR, "gaddy_derived_lexicon.txt"))
    print("using derived lexicon:", der[0])
elif old:                                 # fallback: reuse your existing lexicon under the new name
    shutil.copy(old[0], os.path.join(LM_DIR, "gaddy_derived_lexicon.txt"))
    shutil.copy(old[0], os.path.join(LM_DIR, "gaddy_lexicon.txt"))
    print("derived lexicon not found -> reused gaddy_lexicon.txt under the new name (fallback)")
else:
    print("NO lexicon found anywhere.")

# show how the repo builds the derived lexicon, in case the fallback needs replacing
!grep -rniE "gaddy_derived_lexicon|derived_lexicon|def .*lexicon" /content/silent_speech --include=*.py | head -20
print("KenLM dir now:", os.listdir(LM_DIR))

lm.bin           : ['/content/drive/MyDrive/silent_speech/KenLM/lm.bin', '/content/silent_speech/KenLM/lm.bin']
derived lexicon  : ['/content/silent_speech/KenLM/gaddy_derived_lexicon.txt']
old lexicon      : ['/content/drive/MyDrive/silent_speech/KenLM/gaddy_lexicon.txt']


SameFileError: '/content/silent_speech/KenLM/gaddy_derived_lexicon.txt' and '/content/silent_speech/KenLM/gaddy_derived_lexicon.txt' are the same file

In [ ]:
!cd {REPO_DIR} && CKPT_DIR=/content/ckpts DATA_PATH={DRIVE_PROJECT} \
    python recognition_model.py --model tinymyo --evaluate_saved {CKPT}

In [ ]:
import numpy as np, torch.nn.functional as F, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from hdf5_dataset import H5EmgDataset
from architecture import EMGTransformer

testset = H5EmgDataset(dev=False, test=True)            # loads config/data.yaml (h5_path patched)
print("sample keys:", list(testset[0].keys()))          # confirm 'raw_emg' and 'text'
n_chars = len(testset.text_transform.chars)
model = EMGTransformer(num_features=testset.num_features, num_outs=n_chars + 1,   # == build_model("tinymyo")
                       in_chans=8, embed_dim=192, n_layer=8, n_head=3, mlp_ratio=4.0,
                       attn_drop=0.1, proj_drop=0.1).eval()
sd = torch.load(CKPT, map_location="cpu", weights_only=False)
if isinstance(sd, dict) and "model_state_dict" in sd: sd = sd["model_state_dict"]
model.load_state_dict(sd, strict=True)
print("params:", sum(p.numel() for p in model.parameters()))   # ~4.5M

class Wrap(torch.nn.Module):
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, x): return s.m(x, x, x)                # only x_raw used
FP32 = "/content/model_fp32_tinymyo.onnx"; INT8 = "/content/model_int8_tinymyo.onnx"
ex = testset[0]["raw_emg"].unsqueeze(0)
torch.onnx.export(Wrap(model).eval(), ex, FP32, input_names=["emg"], output_names=["logits"],
                  dynamic_axes={"emg":{0:"batch",1:"time"},"logits":{0:"batch",1:"time"}},
                  opset_version=17, dynamo=False)
quantize_dynamic(FP32, INT8, weight_type=QuantType.QInt8)
mb = lambda p: os.path.getsize(p)/1e6
print(f"FP32 {mb(FP32):.2f} MB -> INT8 {mb(INT8):.2f} MB ({mb(FP32)/mb(INT8):.2f}x)")

In [ ]:
import tqdm, jiwer
from torchaudio.models.decoder import ctc_decoder
def build_decoder(dset, beam=1500):
    tkns = [c for c in dset.text_transform.chars] + ["_"]
    return ctc_decoder(lexicon=os.path.join(LM_DIR,"gaddy_lexicon.txt"), tokens=tkns,
                       lm=os.path.join(LM_DIR,"lm.bin"), blank_token="_", sil_token="|",
                       nbest=1, lm_weight=2, beam_size=beam)
def onnx_wer(path, dset, decoder):
    s = ort.InferenceSession(path, providers=["CPUExecutionProvider"]); nm = s.get_inputs()[0].name
    refs, preds = [], []
    for i in tqdm.tqdm(range(len(dset)), desc=os.path.basename(path)):
        ex = dset[i]; x = ex["raw_emg"].numpy().astype(np.float32)[None, ...]
        y = s.run(None, {nm: x})[0]; logp = F.log_softmax(torch.from_numpy(y), dim=-1)
        pred = dset.text_transform.clean_text(" ".join(decoder(logp)[0][0].words).strip())
        tgt = dset.text_transform.clean_text(ex["text"])
        if tgt != "": refs.append(tgt); preds.append(pred)
    return jiwer.wer(refs, preds)
decoder = build_decoder(testset)
wer_fp32 = onnx_wer(FP32, testset, decoder); wer_int8 = onnx_wer(INT8, testset, decoder)
mb = lambda p: os.path.getsize(p)/1e6
print("========= TINYMYO (8-layer) RESULTS =========")
print(f"FP32 ONNX : WER {wer_fp32*100:6.2f}%   size {mb(FP32):6.2f} MB")
print(f"INT8 ONNX : WER {wer_int8*100:6.2f}%   size {mb(INT8):6.2f} MB   ({mb(FP32)/mb(INT8):.2f}x)")
print(f"RQ1 margin: INT8 <= {wer_fp32*1.10*100:.2f}%  -> {'PASS' if wer_int8<=wer_fp32*1.10 else 'FAIL'}")